In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import preprocessing, features

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')


In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = preprocessing.store_data(store_df)

# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always
train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1) # Not available in test
train_df = features.attach_store_data(train_df, store_df)

test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_22452\2952918199.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1) # Not available in test


In [137]:
""" Feature engineering """

# Competition-related features
train_df['CompetitionDistance'] = train_df['CompetitionDistance'].apply(np.log1p)
train_df['CompetitionSinceMonths'] = ( (train_df['Date'] - train_df['CompetitionSinceDate']).dt.days / 30.0 ).round()

# Promotion related features
train_df['Promo2SinceWeeks'] =  ( (train_df['Date'] - train_df['Promo2SinceDate']).dt.days / 7.0 ).fillna(0).round().astype(int) * train_df['Promo2']

# Basic date features
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week
train_df['Month'] = train_df['Date'].dt.month
train_df['Year'] = train_df['Date'].dt.year
train_df['Quarter'] = train_df['Date'].dt.quarter

# Calendar and seasonality features
train_df['is_weekend'] = train_df['Date'].dt.dayofweek >= 5

# Cyclical features
train_df['Month_sin'] = np.sin(2 * np.pi * train_df['Month'] / 12)
train_df['Month_cos'] = np.cos(2 * np.pi * train_df['Month'] / 12)
train_df['Dayofweek_sin'] = np.sin(2 * np.pi * train_df['DayOfWeek'] / 7)
train_df['Dayofweek_cos'] = np.cos(2 * np.pi * train_df['DayOfWeek'] / 7)

# Lag and rolling features - TODO: We are predicting 6 weeks ahead.
train_df['lag_1'] = train_df['target'].shift(1)
train_df['rolling_mean_7'] = train_df['target'].shift(1).rolling(7).mean()
train_df['rolling_std_7'] = train_df['target'].shift(1).rolling(7).std()

# Drop useless
#train_df.drop(['Promo2SinceDate', 'CompetitionSinceDate'], axis=1, inplace=True)
